# Detecting Motor Anomalies

Preventing motor anomalies is a bit more complicated than battery issues. Usually, motors operate in a certain range of power, but sometimes they may present anomalous behavior. Their power consumption can go to high, due to environmental issues, or too low, due to aging issues.

As usual, let's start by recovering and looking at data:

In [ ]:
%store -r data
data.head()

In [ ]:
%store -r bucket
bucket

# Exploratory Data Analysis

In [ ]:
train_data = data[["motor_peak_mA"]]
train_data = train_data[train_data["motor_peak_mA"] > 0]
train_data.head()

In [ ]:
train_data.describe()

In [ ]:
import matplotlib.pyplot as plt
train_data.plot(rot=30)

## Synthetic Ground Truth

In [ ]:
anomalies = data[["motor_peak_mA"]]
anomalies = anomalies[anomalies["motor_peak_mA"] > 0]
anomalies.info()

In [ ]:
from sklearn.model_selection import train_test_split
train_data, test_dataframe = train_test_split(anomalies, test_size=0.2)

In [ ]:
test_data = test_dataframe.copy()
test_data["anomaly"] = test_data["motor_peak_mA"] > 4000
test_data["anomaly"] = test_data["anomaly"] | (test_data["motor_peak_mA"] > 50) & (test_data["motor_peak_mA"] < 200)
test_data["anomaly"] = test_data["anomaly"].astype(int)
test_data.groupby("anomaly").count().head()

In [ ]:
test_data.describe()

In [ ]:
train_data.describe()

# Random Cut Forest Training

In [ ]:
train_array  = train_data.values
test_array   = test_data[["motor_peak_mA"]].values
labels_array = test_data["anomaly"].values

In [ ]:
import io, struct, boto3

s3bucket = boto3.resource('s3').Bucket(bucket)

def _varint(n):
    buf = b''
    while n > 0x7f:
        buf += bytes([0x80 | (n & 0x7f)])
        n >>= 7
    return buf + bytes([n])

def _map_entry(field_num, key, floats):
    packed = struct.pack('{}f'.format(len(floats)), *floats)
    tensor = b'\x0a' + _varint(len(packed)) + packed
    value  = b'\x12' + _varint(len(tensor)) + tensor
    kv     = b'\x0a' + _varint(len(key)) + key.encode() + b'\x12' + _varint(len(value)) + value
    return _varint((field_num << 3) | 2) + _varint(len(kv)) + kv

def upload_recordio(array, key, labels=None):
    buf = io.BytesIO()
    for i, row in enumerate(array.astype('float32')):
        rec = _map_entry(1, 'values', row.tolist())
        if labels is not None:
            rec += _map_entry(2, 'values', [float(labels[i])])
        buf.write(struct.pack('II', 0xced7230a, len(rec)))
        buf.write(rec)
        buf.write(b'\x00' * ((4 - len(rec) % 4) % 4))
    buf.seek(0)
    s3bucket.Object(key).upload_fileobj(buf)

In [ ]:
import time
import boto3
from sagemaker import get_execution_role, image_uris

region = boto3.Session().region_name
role = get_execution_role()

rcf_container = image_uris.retrieve("randomcutforest", region)

prefix = "mt-motor-anomaly"
train_key = f"{prefix}/input/train.rio"
test_key = f"{prefix}/input/test.rio"

upload_recordio(train_array, train_key)
upload_recordio(test_array, test_key, labels_array)
print(rcf_container)

In [ ]:
import boto3

job_name = "mt-motor-anomaly-" + time.strftime("%Y-%m-%d-%H-%M-%S", time.gmtime())

sm = boto3.client('sagemaker')

sm.create_training_job(
    TrainingJobName=job_name,
    AlgorithmSpecification={
        "TrainingImage": rcf_container,
        "TrainingInputMode": "File",
    },
    RoleArn=role,
    InputDataConfig=[
        {
            "ChannelName": "train",
            "ContentType": "application/x-recordio-protobuf",
            "DataSource": {
                "S3DataSource": {
                    "S3DataType": "S3Prefix",
                    "S3Uri": f"s3://{bucket}/{train_key}",
                    "S3DataDistributionType": "ShardedByS3Key",
                }
            },
        },
        {
            "ChannelName": "test",
            "ContentType": "application/x-recordio-protobuf",
            "DataSource": {
                "S3DataSource": {
                    "S3DataType": "S3Prefix",
                    "S3Uri": f"s3://{bucket}/{test_key}",
                    "S3DataDistributionType": "FullyReplicated",
                }
            },
        },
    ],
    OutputDataConfig={
        "S3OutputPath": f"s3://{bucket}/{prefix}/output",
    },
    ResourceConfig={
        "InstanceType": "ml.m5.large",
        "InstanceCount": 1,
        "VolumeSizeInGB": 10,
    },
    HyperParameters={
        "num_samples_per_tree": "512",
        "num_trees": "50",
        "feature_dim": "1",
        "eval_metrics": "[\"accuracy\"]",
    },
    StoppingCondition={
        "MaxRuntimeInSeconds": 3600,
    },
)

waiter = sm.get_waiter('training_job_completed_or_stopped')
waiter.wait(TrainingJobName=job_name)
training_info = sm.describe_training_job(TrainingJobName=job_name)
print("Training job name:", job_name)
print("Training job status:", training_info["TrainingJobStatus"])


In [ ]:
model_artifacts = training_info["ModelArtifacts"]["S3ModelArtifacts"]
print("Model artifacts:", model_artifacts)


In [ ]:
import boto3

model_name           = job_name + "-model"
endpoint_config_name = job_name + "-epc"
endpoint_name        = job_name + "-ep"

sm = boto3.client('sagemaker')

sm.create_model(
    ModelName=model_name,
    PrimaryContainer={
        "Image": rcf_container,
        "ModelDataUrl": model_artifacts,
    },
    ExecutionRoleArn=role,
)

sm.create_endpoint_config(
    EndpointConfigName=endpoint_config_name,
    ProductionVariants=[{
        "VariantName": "AllTraffic",
        "ModelName": model_name,
        "InitialInstanceCount": 1,
        "InstanceType": "ml.m5.large",
        "InitialVariantWeight": 1,
    }]
)

sm.create_endpoint(
    EndpointName=endpoint_name,
    EndpointConfigName=endpoint_config_name,
)

sagemaker_runtime = boto3.client('sagemaker-runtime')

waiter = sm.get_waiter('endpoint_in_service')
waiter.wait(EndpointName=endpoint_name)
print('Endpoint: ' + endpoint_name)


In [ ]:
rcf_inference_endpoint = endpoint_name
%store rcf_inference_endpoint
rcf_inference_endpoint

In [ ]:
import json

sample_data = test_data[["motor_peak_mA"]].values
csv_body = "\n".join(",".join(str(v) for v in row) for row in sample_data)

res = sagemaker_runtime.invoke_endpoint(
    EndpointName=endpoint_name,
    Body=csv_body,
    ContentType="text/csv",
)
results = json.loads(res["Body"].read().decode("utf-8"))
results


In [ ]:
import pandas as pd
import numpy as np
sigmas = 2

scores = [s["score"] for s in results["scores"]]
series = pd.Series(scores).dropna()
score_mean = series.mean()
score_std = series.std()
score_cutoff = score_mean + sigmas * score_std
print("NaN scores dropped:", len(scores) - len(series))
(score_mean, series.max(), score_std, score_cutoff)


In [ ]:
detected = series[series > score_cutoff]
anomaly_rows = pd.DataFrame({
    "motor_peak_mA": sample_data[detected.index, 0],
    "score": detected.values,
})
print("{} anomalies detected".format(len(anomaly_rows)))
anomaly_rows


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

plot_df = pd.DataFrame({
    "motor_peak_mA": sample_data[:, 0],
    "score": pd.Series(scores).dropna().values,
})
plot_df["anomaly"] = False
plot_df.loc[detected.index, "anomaly"] = True

fig, ax1 = plt.subplots(figsize=(10, 5))
ax1.plot(plot_df.index, plot_df["motor_peak_mA"], label="motor_peak_mA", color="tab:blue")
ax1.scatter(plot_df.loc[plot_df["anomaly"], :].index, plot_df.loc[plot_df["anomaly"], "motor_peak_mA"], 
            color="red", marker="x", s=100, label="anomaly")
ax1.set_xlabel("Sample index")
ax1.set_ylabel("motor_peak_mA", color="tab:blue")
ax1.legend(loc="upper left")
ax2 = ax1.twinx()
ax2.plot(plot_df.index, plot_df["score"], label="anomaly score", color="tab:orange", alpha=0.7)
ax2.axhline(score_cutoff, color="tab:red", linestyle="--", label="score cutoff")
ax2.set_ylabel("score", color="tab:orange")
ax2.legend(loc="upper right")
plt.title("Motor peak vs anomaly score with detected anomalies")
plt.tight_layout()
plt.show()


### Interpreting the anomaly scores
A Random Cut Forest score is higher when a data point is more anomalous relative to the training distribution.
Here we use a simple threshold of mean + 2 standard deviations to flag anomalies.
- `motor_peak_mA` values with scores above `score_cutoff` are treated as likely anomalies.
- If the model returns `NaN` scores, that usually means the input format was invalid or the endpoint could not parse the request.
- We now send the request as newline-separated CSV rows so each measurement is evaluated correctly.

Using `sigmas = 2` reduces the number of flagged anomalies and focuses on the most extreme outliers.
If you want to detect more subtle anomalies, decrease `sigmas` to 1 or 1.5.


## Motor Maintenance

Now that we can detect anomalies in past data, let's combine that with forecasting for predictive [motor maintenance](mt-motor-maintenance.ipynb).